# KMeans Model Testing

# 1. Importing Libraries and the Data

In [14]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error
from sklearn.cluster import KMeans
from collect_log_data import read_log_data

In [15]:
inputs, targets = read_log_data(verbose = True)

Reading prepared data from csv files...
Prepared data collected


# 2. Testing KMeans

Due to the results of the neural network testing, KMeans is unlikely to do well since more complex neural network models tended to do better (more hidden layers), which suggests that this problem is also complex. Since KMeans works best on simple problems with linear relationships, that suggests that KMeans won't have good accuracy. However, the test will be continued because it was promised in the project proposal and it might still give interesting results.

The number of clusters that will be used on the KMeans model will be 3, two classes to represent each team winning and a third class to represent a tie. KMeans will be evaluated by first being trained on the inputs from the data. Then it's classifications for each input will be stored alongside the actual result of the match. Next, the most common value will be chosen from each predicted classification to see what match result (Blu win, Red win, or tie) that classification most likely maps to. Lastly, the accuracy of the model will be measured by the percentage of points in each class have the most common match result in it. If a class has the same most common match result as another class (for example, if 2 classes both map mostly to Blu team winning), extra steps will have to be taken to attempt to interpret what match results the classes map to.

In [22]:
K = 3

In [23]:
kmeans = KMeans(n_clusters = K).fit(inputs)

In [26]:
predictions = kmeans.predict(inputs)

In [29]:
# Map targets (scores of each team) to match results (0 for Red win, 1 for Blu win, 2 for tie)
target_results = []
for target in targets:
	split_scores = np.hsplit(target, 2)
	red_score = np.argmax(split_scores[0], axis = -1)
	blu_score = np.argmax(split_scores[1], axis = -1)

	if red_score > blu_score:
		target_results.append(0)
	elif blu_score > red_score:
		target_results.append(1)
	else:
		target_results.append(2)

print(len(target_results))

23975


In [32]:
classes = [[], [], []]

for i in range(predictions.size):
	classes[predictions[i]].append(target_results[i])

In [33]:
# Got this function to find the most frequent element in a list from GeeksforGeeks
# https://www.geeksforgeeks.org/python/python-find-most-frequent-element-in-a-list/
def most_frequent(l):
	return max(set(l), key = l.count)

In [34]:
# Function to map a class (0, 1, or 2) to a team win string
def map_class_to_result_str(result):
	match result:
		case 0:
			return "Red Win"
		case 1:
			return "Blu Win"
		case 2:
			return "Tie"
		case _:
			raise ValueError("Invalid class (not 0, 1, or 2)")

In [35]:
print("Most common match results for each class")
for i in range(len(classes)):
	result = most_frequent(classes[i])
	result_str = map_class_to_result_str(result)
	print(f"Most frequent for class {i+1}: {result_str}")

Most common match results for each class
Most frequent for class 1: Blu Win
Most frequent for class 2: Blu Win
Most frequent for class 3: Blu Win


This is a problem since all 3 classes map to the same result. It's expected since, according to the exploratory data analysis, Blu team wins are more common than Red team wins. Below is some code to look at the percentage of each match result on each team. If there are any significant differences, then something may be able to be done to further deduce what each class from the model is really mapped to (possibly based on second most common match result). If they're all similar, then that likely means KMeans cannot handle this problem.

In [40]:
for i in range(len(classes)):
	red_wins = classes[i].count(0)
	blu_wins = classes[i].count(1)
	ties = classes[i].count(2)
	red_win_percentage = red_wins / len(classes[i]) * 100
	blu_win_percentage = blu_wins / len(classes[i]) * 100
	tie_percentage = ties / len(classes[i]) * 100
	print(f"Class {i+1}:")
	print(f"Red Wins: {red_win_percentage:.2f}%")
	print(f"Blu Wins: {blu_win_percentage:.2f}%")
	print(f"Ties: {tie_percentage:.2f}%")

Class 1:
Red Wins: 41.50%
Blu Wins: 53.37%
Ties: 5.13%
Class 2:
Red Wins: 42.56%
Blu Wins: 51.40%
Ties: 6.04%
Class 3:
Red Wins: 43.34%
Blu Wins: 51.10%
Ties: 5.56%


All of the classes seems to be made up of nearly the same percentages of match results. Because of this, there is no way to further deduce what actual match results the classes map to and KMeans does not work for this problem.